<a href="https://colab.research.google.com/github/bmulaw/WebScrapping/blob/main/T2_homework_robots_robotfileparser_protego.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework: robots.txt &mdash; RobotFileParser vs Protego

**Name:** Bernard Mulaw  **Date:** 27/10/2026

**Goal:** read `robots.txt`, scrape politely, and see **why `Protego` beats the stdlib `RobotFileParser`**. No BeautifulSoup, no HTML, no crawling.

What you'll test on:
- `quotes.toscrape.com` &mdash; no robots.txt (404 = no rules)
- `scrapethissite.com` &mdash; a small robots.txt with a few `Disallow` lines
- two **literal** robots.txt strings (provided) so the key tasks run offline and identically

> Run top to bottom. Pauses between requests are polite scraping, not bugs.

### What you'll use
- `requests` + the provided `polite_get()` (honest UA, waits &ge; 1 s)
- `RobotFileParser` (stdlib): `set_url`, `parse`, `can_fetch(ua, url)`, `crawl_delay`, `site_maps`, `allow_all` / `disallow_all`
- `Protego`: `Protego.parse`, `can_fetch(url, ua)`, `crawl_delay` &mdash; note the **reversed** `can_fetch` order

### Rules
- Use the helpers below. Don't change them.
- &ge; 1 s between requests. No parallel requests.
- Check robots.txt first (Task 0); never fetch what it disallows.
- Fill every **✍️ Your answer** cell and show your code.
- **Kernel → Restart & Run All** must pass before you submit.

In [ ]:
# Install the libraries this notebook uses (run once per environment).
#   requests - fetch robots.txt over HTTP
#   protego  - a robots.txt parser that understands the * and $ wildcards
%pip install requests protego --quiet

### Helpers (provided &mdash; read them, don't change them)
Imports, a polite GET, `show_robots()` to read a file by eye, `robots_for()` from the demo, and two literal robots.txt strings for the offline tasks.

In [ ]:
import time
from urllib.robotparser import RobotFileParser  # Python's built-in robots.txt reader
import requests                                 # the HTTP client (fetches robots.txt)
from protego import Protego                     # robots.txt parser that handles * and $

UA = "EducationalWebScrapingLab/1.0 (homework)"   # honest name for our scraper
HEADERS = {"User-Agent": UA}
QUOTES = "https://quotes.toscrape.com"            # publishes NO robots.txt (404)
STS = "https://www.scrapethissite.com"            # publishes a robots.txt
_last = [0.0]                                     # timestamp of the previous request

In [ ]:
def polite_get(url, delay=1.0, **kwargs):
    """GET with an honest UA, waiting >= delay seconds since the previous request."""
    wait = delay - (time.time() - _last[0])
    if wait > 0:
        time.sleep(wait)
    try:
        r = requests.get(url, headers=HEADERS, timeout=15, **kwargs)
        return r
    finally:
        _last[0] = time.time()

def show_robots(base_url):
    """Fetch a site's /robots.txt and print its HTTP status and body."""
    url = base_url + "/robots.txt"
    r = polite_get(url)
    print(f"{url}  ->  HTTP {r.status_code}")
    print("------------")
    if r.status_code == 200:
        print(r.text)                          # short enough to read in full
    else:
        print("No robots.txt published (a 404 means 'no rules' - but stay polite anyway).")

def robots_for(base_url):
    """Return a parsed RobotFileParser for a site, read with OUR User-Agent."""
    rp = RobotFileParser()
    rp.set_url(base_url + "/robots.txt")
    try:
        r = requests.get(base_url + "/robots.txt", headers=HEADERS, timeout=15)
    except requests.RequestException:
        rp.allow_all = True                    # couldn't reach it: don't hard-block ourselves
        return rp
    if r.status_code == 200:
        rp.parse(r.text.splitlines())          # 200 -> read the REAL rules
    elif r.status_code in (401, 403):
        rp.disallow_all = True                 # 401/403 -> treat the whole site as blocked
    else:
        rp.allow_all = True                    # 404 etc. -> no rules -> everything allowed
    return rp

# Two LITERAL robots.txt strings so the crawl-delay and wildcard tasks run the
# same way every time, with no network. Don't change them.
DELAY_ROBOTS = """
User-agent: *
Crawl-delay: 5
Disallow: /private/
Sitemap: https://example.com/sitemap.xml
""".strip()

WILDCARD_ROBOTS = """
User-agent: *
Disallow: /*.pdf$
Disallow: /data/
Allow: /data/public/
""".strip()

print("Helpers ready.")

Helpers ready.


---
## Task 0 · Read each robots.txt by eye
`robots.txt` lives at the domain root (`https://site/robots.txt`) and is the first URL a polite scraper requests. Look before you code.

In [ ]:
# print each site's robots.txt by eye, using the provided helper.
#   a) call show_robots(QUOTES)   -> expect HTTP 404 (no file published)
#   b) call show_robots(STS)      -> expect HTTP 200 and a few lines of rules
show_robots(QUOTES)
show_robots(STS)

https://quotes.toscrape.com/robots.txt  ->  HTTP 404
------------
No robots.txt published (a 404 means 'no rules' - but stay polite anyway).
https://www.scrapethissite.com/robots.txt  ->  HTTP 200
------------
User-agent: *
Disallow: /lessons/
Disallow: /faq/


**Questions**
1. Which site returns a **404**, and what does a 404 mean for a scraper's permissions?
2. Copy out the `Disallow` lines that `scrapethissite.com` lists.

<details>
<summary><b>Hint</b></summary>

Check the `HTTP <status>` line. 404 = *no rules published*, not *everything forbidden*. For STS, read the lines under `User-agent: *`.

</details>

✍️ **Your answer:**
1. `quotes.toscrape.com/robots.txt` returned a 404 which means there are no policy stopping web scrapping on that site (scrapers have permission to scrape), but you should still be polite in how you scrape it (ex. waiting 1 second between scrape requests).
2. `scrapethissite.com/robots.txt` returned these Disallow lines: `Disallow: /lessons/`, `Disallow: /faq/`

---
## Task 1 · Build a parser per site and report its flags
`robots_for()` fetches robots.txt with our UA and builds a `RobotFileParser`: 404 &rarr; `allow_all`, 401/403 &rarr; `disallow_all`, 200 &rarr; real rules.

In [ ]:
rp_quotes = robots_for(QUOTES)   # provided: no robots.txt -> allow_all
rp_sts = robots_for(STS)         # provided: has robots.txt -> real rules

# for each parser, print its allow_all and disallow_all flags.
#   e.g. print("quotes allow_all:", rp_quotes.allow_all, "disallow_all:", rp_quotes.disallow_all)
print("quotes allow_all:", rp_quotes.allow_all, "disallow_all:", rp_quotes.disallow_all)
print("------------------------------------------")
print("STS allow_all:", rp_sts.allow_all, "disallow_all:", rp_sts.disallow_all)

quotes allow_all: True disallow_all: False
------------------------------------------
STS allow_all: False disallow_all: False


**Questions**
1. Which site is `allow_all = True`, and why? (Tie it back to the status code from Task 0.)
2. When `allow_all` is `True`, is a URL allowed because a *rule says yes*, or because there are *no rules at all*? Explain the difference.

<details>
<summary><b>Hint</b></summary>

404 &rarr; no file &rarr; `allow_all = True` &rarr; `can_fetch` is `True` by default. Absence of rules, not permission.

</details>

✍️ **Your answer:**
1. Quotes site has `allow_all = True` because this site has no restriction/rule on robots scraping the site (has no `/robots.txt` url since when we try to go to that url, we get a 404).
2. When `allow_all` is `True`, the URL is allowed becasue there are no rules at all. There are no `/robots.txt` URL with a file that specifically says `allow: *`, but if there were, then that that means the rule says yes. Since we don't have that file, there are no rules at all.

---
## Task 2 · `can_fetch` table on scrapethissite.com
Ask `rp_sts.can_fetch(UA, url)` &mdash; order is **(user_agent, url)** &mdash; for each path and print an `ALLOWED` / `BLOCKED` table.

In [ ]:
urls_to_check = [
    STS + "/",
    STS + "/pages/",
    STS + "/pages/simple/",
    STS + "/pages/forms/",
    STS + "/faq/",
    STS + "/lessons/",
    STS + "/lessons/intro/",
]

# loop over urls_to_check and, for each url, print one line like the demo:
#   print(f"{'ALLOWED' if rp_sts.can_fetch(UA, url) else 'BLOCKED':8} {url}")
for url in urls_to_check:
  print(f"{'ALLOWED' if rp_sts.can_fetch(UA, url) else 'BLOCKED':8} {url}")

ALLOWED  https://www.scrapethissite.com/
ALLOWED  https://www.scrapethissite.com/pages/
ALLOWED  https://www.scrapethissite.com/pages/simple/
ALLOWED  https://www.scrapethissite.com/pages/forms/
BLOCKED  https://www.scrapethissite.com/faq/
BLOCKED  https://www.scrapethissite.com/lessons/
BLOCKED  https://www.scrapethissite.com/lessons/intro/


**Questions**
1. Which URLs come back `BLOCKED`, and which exact `Disallow` line blocks each one?
2. Why are the `/pages/...` URLs `ALLOWED`? (Think about how robots.txt matches a path.)

<details>
<summary><b>Hint</b></summary>

robots.txt matches by path **prefix**. `/lessons/intro/` starts with `/lessons/`, so it's blocked. Nothing disallows `/pages/`. Mind the order: `can_fetch(UA, url)`.

</details>

✍️ **Your answer:**
1. BLOCKED URLs are: [https://www.scrapethissite.com/faq/, https://www.scrapethissite.com/lessons/, https://www.scrapethissite.com/lessons/intro/] where `scrapethissite.com/faq` is blocked by the `Disallow: /lessons/`, and `scrapethissite.com/lessons/` and` scrapethissite.com/lessons/intro` are blocked by `Disallow: /faq/` from the `scrapethissite.com/robots.txt` url.
2. `scrapethissite.com/robots.txt` has no specific `ALLOWED` urls, however the directory `/pages` is not directly disallowed, so there are no rules specifically disallowing the websrapper from scrapping the `/pages` URL path and its child paths.

---
## Task 3 · Crawl-delay and sitemaps (politeness)
`crawl_delay(UA)` returns the required wait or `None`; `site_maps()` returns `Sitemap:` URLs or `None`. STS sets neither (both `None`), so parse `DELAY_ROBOTS` to see real values.

In [ ]:
# Part A - the REAL site (expect None / None):
print(rp_sts.crawl_delay(UA))
print(rp_sts.site_maps())

# Part B - a LITERAL file that DOES set them (expect 5 and the example sitemap):
rp2 = RobotFileParser()
rp2.parse(DELAY_ROBOTS.splitlines())   # provided
print(rp2.crawl_delay(UA))
print(rp2.site_maps())

None
None
5
['https://example.com/sitemap.xml']


**Questions**
1. What are STS's real crawl-delay and sitemap values, and what does `None` tell you about how long you should wait between requests?
2. In words, how should a `Crawl-delay: 5` change the pause in your request loop compared with your default?

<details>
<summary><b>Hint</b></summary>

`None` &rarr; keep your polite default (&ge; 1 s). `Crawl-delay: 5` &rarr; wait at least 5 s. Always honour the larger number.

</details>

✍️ **Your answer:**
1. STS's real crawl-delay and sitemap values are None, which tells me that I should wait 1 second or more between requests (which is the default polite value).
2. A crawl-delay of 5 seconds tells me I should pause 5 seconds between requests to scrape a site instead of 1 seconds. I should always set my delay between webscrapping requests to the maximum value between 1 second and the site's crawl-delay.

---
## Task 4 · The same robots.txt through Protego
`Protego` follows RFC 9309 and is what Scrapy uses. Parse the SAME STS rules and compare. **Mind the reversed argument order:**

| Parser | `can_fetch` call | First argument |
|---|---|---|
| `RobotFileParser` | `rp_sts.can_fetch(UA, url)` | **user-agent** |
| `Protego` | `rules.can_fetch(url, UA)` | **url** |

Swap them and you get wrong answers, silently.

In [ ]:
# Fetch the STS robots.txt text once (politely) and parse it with Protego.
text = polite_get(STS + "/robots.txt").text   # provided
rules = Protego.parse(text)

# 1. loop over the SAME urls_to_check from Task 2 and print Protego's verdict.
#   REMEMBER the reversed order: rules.can_fetch(url, UA)
#   print(f"{'ALLOWED' if rules.can_fetch(url, UA) else 'BLOCKED':8} {url}")
# 2. also print rules.crawl_delay(UA)
for url in urls_to_check:
  print(f"{'ALLOWED' if rules.can_fetch(url, UA) else 'BLOCKED':8} {url}")
print("crawl delay:", rules.crawl_delay(UA))

ALLOWED  https://www.scrapethissite.com/
ALLOWED  https://www.scrapethissite.com/pages/
ALLOWED  https://www.scrapethissite.com/pages/simple/
ALLOWED  https://www.scrapethissite.com/pages/forms/
BLOCKED  https://www.scrapethissite.com/faq/
BLOCKED  https://www.scrapethissite.com/lessons/
BLOCKED  https://www.scrapethissite.com/lessons/intro/
crawl delay: None


**Questions**
1. Do Protego and RobotFileParser agree on every URL for STS? Why would you expect that here? (Think about whether these rules use any wildcards.)
2. Write out both `can_fetch` signatures and say which argument comes first in each. Which one would break silently if you swapped the arguments?

<details>
<summary><b>Hint</b></summary>

STS uses plain prefix rules (no `*`/`$`), so both agree. The gap shows up only with wildcards &mdash; Task 5. `RobotFileParser.can_fetch(ua, url)` vs `Protego.can_fetch(url, ua)`.

</details>

✍️ **Your answer:**
1. Yes both Protego and RobotFileParser agree on every URL for STS which is expected since they both are validating on the same robots.txt file that does not have any wildcards that we see in Task 5.
2. RobotFileParser has a signature of `can_fetch(self, useragent, url)` with UA coming first and URL coming second with no guarantee the output is a boolean, however Protego has a signature of `can_fetch(self, url: str, user_agent: str) -> bool` with URL being first argument and UA being the second argument. **They both would break silently** when the arguments were swapped, both giving a false value of True for if the robot can fetch the URL

---
## Task 5 · Wildcards: where Protego wins
This is the point of the homework. The stdlib `RobotFileParser` (Python &le; 3.13) has two bugs:
1. it ignores `*` and `$` wildcards, and
2. it takes the *first* matching rule, not the *longest* (RFC 9309 says longest wins).

`Protego` gets both right. Feed `WILDCARD_ROBOTS` to both parsers (no network) and watch them disagree:

```
User-agent: *
Disallow: /*.pdf$
Disallow: /data/
Allow: /data/public/
```

In [ ]:
BASE = "https://example.com"

rp_w = RobotFileParser()
rp_w.parse(WILDCARD_ROBOTS.splitlines())   # provided: stdlib parser
rules_w = Protego.parse(WILDCARD_ROBOTS)   # provided: Protego parser

wildcard_urls = [
    BASE + "/reports/2024.pdf",
    BASE + "/reports/2024.pdf?dl=1",
    BASE + "/data/public/report.csv",
    BASE + "/data/secret.csv",
]

# TODO: for each url, print the stdlib verdict and the Protego verdict side by side.
#   stdlib:  rp_w.can_fetch(UA, url)      (order: user_agent, url)
#   protego: rules_w.can_fetch(url, UA)   (order: url, user_agent)
# Suggested line:
#   std = 'ALLOWED' if rp_w.can_fetch(UA, url) else 'BLOCKED'
#   pro = 'ALLOWED' if rules_w.can_fetch(url, UA) else 'BLOCKED'
#   print(f"stdlib={std:8} protego={pro:8} {url}")
for url in wildcard_urls:
  stdlib = 'ALLOWED' if rp_w.can_fetch(UA, url) else 'BLOCKED'
  protego = 'ALLOWED' if rules_w.can_fetch(url, UA) else 'BLOCKED'
  print(f"stdlib={stdlib:8} protego={protego:8} {url}")


stdlib=ALLOWED  protego=BLOCKED  https://example.com/reports/2024.pdf
stdlib=ALLOWED  protego=ALLOWED  https://example.com/reports/2024.pdf?dl=1
stdlib=BLOCKED  protego=ALLOWED  https://example.com/data/public/report.csv
stdlib=BLOCKED  protego=BLOCKED  https://example.com/data/secret.csv


In [ ]:
import platform
print(platform.python_version())

3.12.12


**Questions**
1. For which two URLs do the parsers disagree? For each, which parser is correct per RFC 9309, and which rule should apply?
2. Given those disagreements, why is Protego the safer choice for real-world robots.txt files?

<details>
<summary><b>Hint</b></summary>

`/reports/2024.pdf`: `Disallow: /*.pdf$` should block it, but stdlib ignores `*`/`$` and allows it. `/data/public/report.csv`: `Allow: /data/public/` is the longer match (allowed), but stdlib takes the first `Disallow: /data/` and blocks it.

</details>

✍️ **Your answer:**
*Noting that I had to use an older version of Python to make this case work as in the latest update of Python, RFP is now capable of wildcard support*

1. The two parsers disagree on URL *https://example.com/reports/2024.pdf* and *https://example.com/data/public/report.csv*. The robots.txt rule explicity disallows url paths matching `/*.pdf$`, which `/reports/2024.pdf` clearly meets since it ends with .pdf and matches with all the texts before .pdf (i.e. the \*) and **should be disallowed**; and the parser **should also be allowed to scrape url paths matching `/data/public/`** since the rule allowing the parsing of `/data/public/` is longer than the other rule disallowing parsing `/data/` (according to RFC 9309), and we can see that the second URL ending in *data/public/report.csv* clearly meets the case. **Thereofre, the Protego parser is correct per RFC 9309**.  
2.Protego is the safer choice for real-world robots.txt since it has the ability to correctly match the url paths with wildcards and because it takes the longest path rule when there's a conflicting allow/disallow rule to similar paths (which is standard to RFC 9309 policy).  

---
## Task 6 · The Protego pipeline (your reusable tool)
Put it all together into one small tool you'll reuse for real scraping: a **Protego gate**. It (1) reads a site's robots.txt **once**, (2) answers `allowed(url)` with Protego, and (3) `get(url)` **only** when allowed, waiting the crawl-delay first. Built on Protego &mdash; not the stdlib parser &mdash; so wildcards and longest-match are handled correctly.

The gate is **provided below** (read it). Your job is to *use* it.

In [ ]:
class ProtegoGate:
    """A polite fetch pipeline built on Protego: ask permission, wait, then fetch."""
    def __init__(self, base_url, default_delay=1.0):
        self.base = base_url.rstrip("/")
        r = polite_get(self.base + "/robots.txt")          # read robots.txt ONCE
        if r.status_code == 200:
            self.rules = Protego.parse(r.text)             # real rules (wildcards + longest-match)
        elif r.status_code in (401, 403):
            self.rules = Protego.parse("User-agent: *\nDisallow: /")   # forbidden -> block all
        else:
            self.rules = Protego.parse("")                 # 404 etc. -> no rules -> allow all
        site_delay = self.rules.crawl_delay(UA) or 0       # site's Crawl-delay, or 0 if none
        self.delay = max(default_delay, site_delay)        # honour the LARGER wait

    def allowed(self, url):
        """True if robots.txt (via Protego) permits fetching url. Order: (url, UA)."""
        return self.rules.can_fetch(url, UA)

    def get(self, url):
        """Fetch url ONLY if allowed, waiting the crawl-delay first. Returns Response or None."""
        if not self.allowed(url):
            print(f"BLOCKED by robots.txt -> {url}")
            return None
        return polite_get(url, delay=self.delay)           # reuses the polite helper

print("Pipeline ready. Build one with:  gate = ProtegoGate(STS)")

Pipeline ready. Build one with:  gate = ProtegoGate(STS)


**Use the gate on `scrapethissite.com`.** Build one, check a few URLs, then fetch an allowed page and watch a blocked one get refused *without* a request.

In [ ]:
gate = ProtegoGate(STS)                       # provided: reads robots.txt once, picks up the crawl-delay
print("delay this gate honours:", gate.delay, "s")

# (a): print ALLOWED/BLOCKED for a few URLs using gate.allowed(url).
#   try: STS + "/pages/", STS + "/lessons/", STS + "/faq/"
#   print(f"{'ALLOWED' if gate.allowed(u) else 'BLOCKED':8} {u}")
for u in [STS + "/pages/", STS + "/lessons/", STS + "/faq/"]:
  print(f"{'ALLOWED' if gate.allowed(u) else 'BLOCKED':8} {u}")

# (b): fetch ONE allowed page and print its status code.
#   r = gate.get(STS + "/pages/"); print("status:", r.status_code if r else "refused")
# (c): call gate.get(STS + "/lessons/") and confirm it returns None (refused, never fetched).
r = gate.get(STS + "/pages/")
print("status for", (STS + "/pages/:"), r.status_code if r else "refused")

stsLessons = gate.get(STS + "/lessons/")
print("ProtegoGate get status for", (STS + "/lessons/:"), stsLessons.status_code if stsLessons else "refused")

delay this gate honours: 1.0 s
ALLOWED  https://www.scrapethissite.com/pages/
BLOCKED  https://www.scrapethissite.com/lessons/
BLOCKED  https://www.scrapethissite.com/faq/
status for https://www.scrapethissite.com/pages/: 200
BLOCKED by robots.txt -> https://www.scrapethissite.com/lessons/
ProtegoGate get status for https://www.scrapethissite.com/lessons/: refused


**Questions**
1. When you call `gate.get()` on a disallowed URL, does it make an HTTP request? Why is that the whole point of the pipeline?
2. The gate uses Protego, not `RobotFileParser`. On a site whose robots.txt uses `*`/`$` wildcards, why does that choice matter?

<details>
<summary><b>Hint</b></summary>

`get()` checks `allowed()` first and returns `None` without fetching when blocked &mdash; so you never hit a disallowed URL. Protego reads wildcards and longest-match correctly, so the gate's permission answers are right where the stdlib parser would be wrong.

</details>

✍️ **Your answer:**

1. All in all, no, `gate.get()` does NOT make an HTTP request to a disallowed URL which is important since we want to respect the rules provided by the website's robots.txt and avoid unnecssary network calls making the website server busy blocking our request when we've known ahead of time that that was going to happen.

   We can see that the call to `gate.get()` on `scrapethissite.com/lesson` will call on method `allowed()` which calls Protego's `can_fetch` function. That function (I looked around at the _protego.py code) checks if the url is a `/robots.text` (which if it was, then would return true), if not, it checks if there's a rule set to the matched url path (if there is not a rule set for the url path, then it returns true for can_fetch), lastly, if there is a rule matched to the given url path, it simply returns true/false based on the rule it cached for the matched url given (I assume its a cache since there's a comment on the code defining it as a "memoization table" with UA as key and a private library called Ruleset as value, which I believe would contain information of disallowed paths specified by robots.txt). In this scenario, the path `/lesson` clearly was `DISALLOW` so I would image the return is to match DISALLOW to False. The false returned from Protego's `can_fetch` then trickles back to the gate method `allow()` mentioned earlier, which then handles that return by also returning False to the caller `gate.get()` that then returns None instead of making a polite get to the given URL.

2. The gate using Protego instead of `RobotFileParser` makes sense because it reads `*`/`$` wildcards in robots.txt and matches given paths correctly with those wildcard rules. This matters because parsing a site whose robots.txt uses `*`/`$` wildcards with RobotFileParser, which (in an older version) does not read wildcards correctly, **we could make wrong decisions on how we scrape the site** (as highlighted in Task 5).

---
## Task 7 · Reflection
2–3 sentences each, no code.

1. When do you trust `Protego` over the stdlib `RobotFileParser`, and why?
2. Why read `robots.txt` *first*, before fetching any page?

✍️ **Your answer:**
1. I trust `Protego` over `RobotFileParser` whenever there are wildcards in the website's robots.txt file defining the paths that are allowed/disallowed to be scrapped. I would also consider Protego more if I notice a site has a conflicting/matching paths where I need to prioritize the allow/disallow rule with the longest path rather than the first-matching rule. All in all, I'd prefer to use Protego since it has a better chance of being RFC 9309 compliant than RobotFileParser, and I want to avoid mistakenly attempting to crawl/scrape what I'm not allowed to (or missing out on crawling/scraping a path that I am allowed to).
2. It's important to read `robots.txt` *first* before fetching any page to be compliant to the website's rule and avoid being blocked from scrapping the contents/data of the site simply because I did not follow the rules. Reading the rules first allows me (no pun intended) to set up my code to fetch what I'm allowed to and get the data I need without worrying about being blocked/denied.

---
### Submission checklist
- [x] Every code cell runs (no `# your code here` left empty)
- [x] Every ✍️ answer cell is filled in
- [x] Honest User-Agent used and the &ge; 1 s delay left in place (no parallel requests)
- [x] **Kernel → Restart & Run All** completes with no errors